# ATAC-seq standard protocol

In [ ]:
#!pip install bioat -U

In [1]:
import os
import json
from glob import glob
from bioat.lib.libpath import HOME
from pprint import pprint

## 参数设置

In [2]:
thread = 20

## 生成samples.json

In [3]:
# project = "20251108"
project = "20251211"

In [4]:
#ls = glob("./data/20251107/*.fastq.gz")
#ls = glob("/media/desk16/iyun3022/00.Project/03.ATAC/00.raw/Act_*.fq.gz")
ls = glob("./data/20251211/CRR*.fq.gz")
ls.sort()
assert ls != []  # 需要非空
ls_se = [i for i in ls if i.endswith("SE.fq.gz")]
ls_pe = [i for i in ls if i.endswith("R1.fq.gz")]

In [5]:
if ls_se:
    ls_sample = [i.split("/")[-1].split("_SE.fq")[0] for i in ls_se]
    end_type = "SE"

if ls_pe:
    ls_sample = [i.split("/")[-1].split("_R1.fq")[0] for i in ls_pe]
    end_type = "PE"

In [ ]:
# Adapters
# Check from MultiQC results


# Nextera Transposase Adapters
# The transposase adapters are used for Nextera tagmentation.
# Read 1
# 5′ TCGTCGGCAGCGTCAGATGTGTATAAGAGACAG
# Read 2
# 5′ GTCTCGTGGGCTCGGAGATGTGTATAAGAGACAG
#R1_AD = 'TCGTCGGCAGCGTCAGATGTGTATAAGAGACAG'
#R2_AD = 'GTCTCGTGGGCTCGGAGATGTGTATAAGAGACAG'

In [1]:
# Already use fastp to auto trim adapter! Defualt no need adapter sequence

In [6]:
def check_genome(x):
    if x == "mm10":
        tsv_path = './ref_data/mm10/mm10.tsv'
    elif x == "hg38":
        tsv_path = './ref_data/hg38/hg38.tsv'
    else:
        raise ValueError(f"Unsupported genome: {x}")
    return tsv_path

In [7]:
# genome = "mm10"
genome = "hg38"

check_genome(genome)

'./ref_data/hg38/hg38.tsv'

In [8]:
def load_tsv(tsv_path):
    d = {}
    with open(tsv_path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            key, value = line.split("\t", 1)
            d[key] = value
    return d


def build_dt(tsv_path, seq_mode, samples, thread, project):
    d = load_tsv(tsv_path)

    # bowtie2 index prefix (remove .tar.gz)
    #bowtie2_index_genome_prefix = d["bowtie2_idx_tar"].replace(".tar.gz", "")
    #bowtie2_index_chrM_prefix = d["bowtie2_mito_idx_tar"].replace(".tar.gz", "")

    # 解压后的索引目录
    bowtie2_index_genome_dir = d["bowtie2_idx_tar"].replace(".tar.gz", "")
    bowtie2_index_chrM_dir   = d["bowtie2_mito_idx_tar"].replace(".tar.gz", "")

    # 假设索引前缀跟目录名一致
    bowtie2_index_genome_prefix = os.path.join(bowtie2_index_genome_dir,
                                               os.path.basename(bowtie2_index_genome_dir))
    bowtie2_index_chrM_prefix   = os.path.join(bowtie2_index_chrM_dir,
                                               os.path.basename(bowtie2_index_chrM_dir))


    dt = {
        "seq_mode": seq_mode,
        "samples": samples,
        "thread": thread,
        "project": project,

        # genome fasta
        "genome": d["ref_fa"],
        "genome_fai": d["ref_fa"][:-3] + ".fai",

        # bowtie2 index prefix (无需加后缀，会自动匹配 .1.bt2/.2.bt2/etc.)
        "bowtie2_index_genome": bowtie2_index_genome_prefix,
        "bowtie2_index_chrM": bowtie2_index_chrM_prefix,

        # 这里你可以继续保留质控用的黑名单等
        "blacklist": d.get("blacklist_unzip", None),
        "tss": d.get("tss_extend2kb", None),

        # macs2 gsize
        "macs2_gsize": d["gensz"]
    }
    return dt

In [9]:
dt = build_dt(check_genome(genome), seq_mode=end_type, samples=ls_sample, thread=thread, project = project)

pprint(dt)

{'blacklist': '/media/desk16/MaTianyu/Pipeline/snakepipes_ATAC-seq/ref_data/hg38/ataqc/hg38.blacklist.bed',
 'bowtie2_index_chrM': '/media/desk16/MaTianyu/Pipeline/snakepipes_ATAC-seq/ref_data/hg38/bowtie2_index/GRCh38_no_alt_analysis_set_GCA_000001405.15.chrM.fa/GRCh38_no_alt_analysis_set_GCA_000001405.15.chrM.fa',
 'bowtie2_index_genome': '/media/desk16/MaTianyu/Pipeline/snakepipes_ATAC-seq/ref_data/hg38/bowtie2_index/GRCh38_no_alt_analysis_set_GCA_000001405.15.fasta/GRCh38_no_alt_analysis_set_GCA_000001405.15.fasta',
 'genome': '/media/desk16/MaTianyu/Pipeline/snakepipes_ATAC-seq/ref_data/hg38/GRCh38_no_alt_analysis_set_GCA_000001405.15.fasta.gz',
 'genome_fai': '/media/desk16/MaTianyu/Pipeline/snakepipes_ATAC-seq/ref_data/hg38/GRCh38_no_alt_analysis_set_GCA_000001405.15.fasta.fai',
 'macs2_gsize': 'hs',
 'project': '20251211',
 'samples': ['CRR118885'],
 'seq_mode': 'PE',
 'thread': 20,
 'tss': '/media/desk16/MaTianyu/Pipeline/snakepipes_ATAC-seq/ref_data/hg38/ataqc/hg38.tss_extend

In [10]:
with open("./samples.json", "wt") as f:
    f.write(json.dumps(dt))

In [ ]:
#MACS2_GSIZE = 'hs'

# Effective genome size. It can be 1.0e+9 or 1000000000, or shortcuts:'hs' for human (2.7e9),
# 'mm' for mouse (1.87e9), 'ce' for C. elegans (9e7) and 'dm' for fruitfly (1.2e8), Default:hs

In [ ]:
dt = {
    "seq_mode": end_type, 
    "samples": ls_sample, 
    "thread": thread,
    #"r1_ad": R1_AD,
    #"r2_ad": R2_AD,
    "genome": f"{HOME}/1.database/db_genomes/genome_fa/genome_ucsc_hg38/genome_ucsc_hg38.fa",
    "bowtie2_index_genome": f"{HOME}/1.database/db_genomes/genome_fa/genome_ucsc_hg38/genome_ucsc_hg38.fa.bowtie2_index",
    "bowtie2_index_chrM": f"{HOME}/1.database/db_genomes/genome_fa/genome_ucsc_hg38/chrM.fa.bowtie2_index",
    "bowtie2_index_plasmid": "ref_data/ref_plasmid/plasmid.fa.bowtie2_index",
    "macs2_gsize": MACS2_GSIZE
}
pprint(dt)

{'bowtie2_index_chrM': '/lustre1/chengqiyi_pkuhpc/zhaohn/1.database/db_genomes/genome_fa/genome_ucsc_hg38/chrM.fa.bowtie2_index',
 'bowtie2_index_genome': '/lustre1/chengqiyi_pkuhpc/zhaohn/1.database/db_genomes/genome_fa/genome_ucsc_hg38/genome_ucsc_hg38.fa.bowtie2_index',
 'bowtie2_index_plasmid': 'ref_data/ref_plasmid/plasmid.fa.bowtie2_index',
 'genome': '/lustre1/chengqiyi_pkuhpc/zhaohn/1.database/db_genomes/genome_fa/genome_ucsc_hg38/genome_ucsc_hg38.fa',
 'macs2_gsize': 'hs',
 'r1_ad': 'TCGTCGGCAGCGTCAGATGTGTATAAGAGACAG',
 'r2_ad': 'GTCTCGTGGGCTCGGAGATGTGTATAAGAGACAG',
 'samples': ['ATACSeq_GFP-NLS_REP-1',
             'ATACSeq_GFP_REP-1',
             'ATACSeq_ND6-DddAwt_REP-1',
             'ATACSeq_ND6-DddAwt_REP-2',
             'ATACSeq_SIRT6-DddA11_REP-1',
             'ATACSeq_SIRT6-DddA11_REP-2',
             'test'],
 'seq_mode': 'PE',
 'thread': 20}


In [9]:
with open("./samples.json", "wt") as f:
    f.write(json.dumps(dt))